In [1]:
"""
Fine-tuning BioBERT on BC5CDR for Named Entity Recognition (NER).

This script performs the following steps:
1. Loads the 'tner/bc5cdr' dataset.
2. Initializes the BioBERT tokenizer and model for Token Classification.
3. Aligns BIO tags with sub-tokens generated by the BERT tokenizer.
4. Sets up the Hugging Face Trainer with optimal hyperparameters for NER.
5. Executes the training loop.

Note: Sub-tokens created by WordPiece (e.g., "aspirin" -> "as", "##pirin") 
are handled by assigning the original label to the first sub-token and 
-100 to subsequent sub-tokens to ignore them in the loss calculation.
"""

'\nFine-tuning BioBERT on BC5CDR for Named Entity Recognition (NER).\n\nThis script performs the following steps:\n1. Loads the \'tner/bc5cdr\' dataset.\n2. Initializes the BioBERT tokenizer and model for Token Classification.\n3. Aligns BIO tags with sub-tokens generated by the BERT tokenizer.\n4. Sets up the Hugging Face Trainer with optimal hyperparameters for NER.\n5. Executes the training loop.\n\nNote: Sub-tokens created by WordPiece (e.g., "aspirin" -> "as", "##pirin") \nare handled by assigning the original label to the first sub-token and \n-100 to subsequent sub-tokens to ignore them in the loss calculation.\n'

In [8]:
"""
96 minutes
Fine-tuning BioBERT on BC5CDR for Named Entity Recognition (NER).

STABILITY FIXES:
1. Uses BertTokenizer explicitly (bypasses tokenizer backend issues).
2. Uses BertForTokenClassification explicitly (bypasses model_type config issues).
3. Adds force_download and revision to ensure clean model files.
"""

import torch
from datasets import load_dataset
# Using explicit BERT classes instead of Auto classes
from transformers import (
    BertTokenizer, 
    BertForTokenClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForTokenClassification,
    BertConfig
)

# 1. Load the dataset
dataset = load_dataset("tner/bc5cdr", split="train", trust_remote_code=True)

# 2. Define Model and Tokenizer
model_checkpoint = "dmis-lab/biobert-base-cased-v1.1"

# Load Tokenizer explicitly
tokenizer = BertTokenizer.from_pretrained(model_checkpoint)

# Label mapping
label_list = ["O", "B-Chemical", "B-Disease", "I-Disease", "I-Chemical"]
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# 3. BIO-tagging Alignment Function
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True,
        max_length=128 # BioBERT performs best with standard lengths
    )

    labels = []
    for i, label in enumerate(examples["tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 4. Processing and Data Insight
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

print("\n--- Data Alignment Insight ---")
example_tokens = tokenizer.convert_ids_to_tokens(tokenized_dataset[0]["input_ids"])
for tok, lab in zip(example_tokens[:10], tokenized_dataset[0]["labels"][:10]):
    print(f"{tok:15} : {id2label[lab] if lab != -100 else 'IGN'}")

# 5. Initialize Model (Explicitly as BertForTokenClassification)
# We use force_download=True once to clear any corrupted local files
model = BertForTokenClassification.from_pretrained(
    model_checkpoint, 
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    force_download=False # Change to True if the error persists one more time
)

# 6. Data Collator - Ensure the tokenizer is passed here
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 7. Training Arguments
training_args = TrainingArguments(
    output_dir="./biobert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=12,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none" 
)

# 8. Initialize Trainer 
# Note: We removed the standalone 'tokenizer' argument to avoid the TypeError
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# 9. Start Training
print("\nStarting Training...")
trainer.train()
print("Success!")


--- Data Alignment Insight ---
[CLS]           : IGN
na              : B-Chemical
##lo            : IGN
##xon           : IGN
##e             : IGN
reverse         : O
##s             : IGN
the             : O
anti            : O
##hy            : IGN


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4251.73it/s]
BertForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight          


Starting Training...


Step,Training Loss
50,0.464285
100,0.159190
150,0.121177
200,0.096989
250,0.076770
300,0.084582
350,0.073352
400,0.077641
450,0.066191
500,0.045535


Success!


In [14]:
## Task B2
import json
import torch

def bio_to_char_spans(raw_text, tokens, predicted_label_ids, id2label):
    entities = []
    current_entity = None
    current_char_pos = 0
    
    for i, (token, label_id) in enumerate(zip(tokens, predicted_label_ids)):
        label_str = id2label[label_id]
        clean_token = token.replace("##", "")
        
        start_idx = raw_text.lower().find(clean_token.lower(), current_char_pos)
        if start_idx == -1:
            start_idx = current_char_pos
        end_idx = start_idx + len(clean_token)
        current_char_pos = end_idx

        if label_str == "O":
            if current_entity:
                entities.append(current_entity)
                current_entity = None
            continue

        entity_type = label_str.split("-")[1]

        # THE MERGE CONDITION:
        # 1. We have a current entity of the same type
        # 2. AND (It's an I-tag OR the gap between them is just spaces/hyphens)
        if current_entity and current_entity["label"] == entity_type:
            gap = raw_text[current_entity["end"]:start_idx]
            # If gap is empty, just spaces, or just a hyphen, merge them
            if label_str.startswith("I-") or gap.strip() in ["", "-"]:
                current_entity["end"] = end_idx
                current_entity["text"] = raw_text[current_entity["start"]:end_idx]
                continue

        # If we couldn't merge, save the old one and start a new one
        if current_entity:
            entities.append(current_entity)
        
        current_entity = {
            "label": entity_type,
            "text": raw_text[start_idx:end_idx],
            "start": start_idx,
            "end": end_idx
        }
                
    if current_entity:
        entities.append(current_entity)
    return entities



def run_task_b2_inference(input_file, output_file, model, tokenizer, id2label):
    """
    Loads data, performs model inference, converts to character spans, and saves JSON.
    """
    # 1. Load the test file
    with open(input_file, 'r') as f:
        test_data = json.load(f)
    
    final_results = []
    
    # Set model to evaluation mode (important for consistent results)
    model.eval()
    
    print(f"Processing {input_file}...")
    
    for entry in test_data:
        doc_id = entry["doc_id"]
        raw_text = entry["text"]
        
        # 2. Tokenize and prepare for model
        # We must keep tokens to pass to our bio_to_char_spans function
        inputs = tokenizer(raw_text, return_tensors="pt", truncation=True, max_length=128)
        
        # Move inputs to same device as model (GPU or CPU)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        # 3. Model Inference (Prediction)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            # Get the index of the highest score for each token
            predictions = torch.argmax(logits, dim=-1).squeeze().tolist()
        
        # Convert IDs back to raw tokens from the tokenizer
        tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze().tolist())
        
        # 4. Filter out special tokens ([CLS], [SEP]) before mapping
        # This keeps the tokens and predictions aligned
        filtered_tokens = []
        filtered_preds = []
        for tok, pred in zip(tokens, predictions):
            if tok not in [tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token]:
                filtered_tokens.append(tok)
                filtered_preds.append(pred)
        
        # 5. Convert BIO tags to Character Spans using your function
        predicted_entities = bio_to_char_spans(raw_text, filtered_tokens, filtered_preds, id2label)
        
        # 6. Append to final list in Stella's "Contract" format
        final_results.append({
            "doc_id": doc_id,
            "predicted_entities": predicted_entities
        })

    # 7. Save to file
    with open(output_file, 'w') as f:
        json.dump(final_results, f, indent=2)
    
    print(f"Done! Results saved to {output_file}")



In [15]:
# 8 minutes
# Part 1: BioBERT Results
run_task_b2_inference("test_easy.json", "biobert_predictions_easy.json", model, tokenizer, id2label)
run_task_b2_inference("test_hard.json", "biobert_predictions_hard.json", model, tokenizer, id2label)


Processing test_easy.json...
Done! Results saved to biobert_predictions_easy.json
Processing test_hard.json...
Done! Results saved to biobert_predictions_hard.json
